##### ### The University of Melbourne, School of Computing and Information Systems
# COMP30027 Machine Learning, 2026 Semester 1

## Assignment 1: Income Classification with Naïve Bayes


**Student ID(s):**     `PLEASE ENTER YOUR ID(S) HERE`


This iPython notebook is a template which you will use for your Assignment 1 submission.

**NOTE: YOU SHOULD ADD YOUR RESULTS, GRAPHS, AND FIGURES FROM YOUR OBSERVATIONS IN THIS FILE TO YOUR REPORT (the PDF file).** Results, figures, etc. which appear in this file but are NOT included in your report will not be marked.

**Adding proper comments to your code is MANDATORY. **

In [1]:
# Import dependencies 
import numpy as np
import pandas as pd
from typing import List
from scipy.stats import norm
from sklearn.naive_bayes import GaussianNB, CategoricalNB
from sklearn.preprocessing import OrdinalEncoder

import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load datasets
adult_train = pd.read_csv('data/adult_supervised_train.csv')
adult_test = pd.read_csv('data/adult_test.csv')
adult_unlab = pd.read_csv('data/adult_unlabelled.csv')
adult_unlab_w_label = pd.read_csv('data/adult_unlabelled_with_labels.csv')

In [3]:
adult_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 16280 entries, 0 to 16279
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             16280 non-null  int64
 1   workclass       15355 non-null  str  
 2   fnlwgt          16280 non-null  int64
 3   education       16280 non-null  str  
 4   education-num   16280 non-null  int64
 5   marital-status  16280 non-null  str  
 6   occupation      15351 non-null  str  
 7   relationship    16280 non-null  str  
 8   race            16280 non-null  str  
 9   sex             16280 non-null  str  
 10  capital-gain    16280 non-null  int64
 11  capital-loss    16280 non-null  int64
 12  hours-per-week  16280 non-null  int64
 13  native-country  15990 non-null  str  
 14  income          16280 non-null  str  
dtypes: int64(6), str(9)
memory usage: 1.9 MB


In [4]:
# Drop fnlwgt (census sampling weight, not a predictive feature)

for df in [adult_train, adult_test, adult_unlab, adult_unlab_w_label]:
    df.drop(columns=['fnlwgt'], inplace=True)

In [5]:
# Split atributes and concept
TARGET = 'income'
y = adult_train[TARGET]
X = adult_train.drop(columns=[TARGET])

In [6]:
# Identify feature types

cat_feats = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_feats = X.select_dtypes(include=['number']).columns.tolist()

print(f"Categorical features: \n{cat_feats}\n")
print(f"Numeric/continuous features: \n{num_feats}")

Categorical features: 
['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']

Numeric/continuous features: 
['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']


In [7]:
# Drop rows with missing values from training data
adult_train_clean = adult_train.dropna(subset=X.columns)

In [8]:
# Assess class distribution
print(f"Number of training rows: {adult_train.shape[0]} -> {adult_train_clean.shape[0]} after dropping missing")
adult_train_clean[TARGET].value_counts(normalize=True)

Number of training rows: 16280 -> 15076 after dropping missing


income
<=50K    0.754112
>50K     0.245888
Name: proportion, dtype: float64

## 1. Supervised model training


In [9]:
# Split attributes and concept on clean data
y_train = adult_train_clean[TARGET].map({'<=50K': 0, '>50K': 1})
X_train = adult_train_clean.drop(columns=TARGET)

In [ ]:
# MixedNB model - part Gaussian, part categorical
class MixedNB:
    def __init__(
            self,
            alpha: float = 1.0, 
    ) -> None:
        # alpha: Laplace smoothing for CategoricalNB
        self.alpha = alpha

    def fit(
            self,
            X: pd.DataFrame, 
            y: pd.Series, 
            sample_weight=None,
    ) -> "MixedNB":
        # Identify feature types from the DataFrame dtypes
        self.cat_feats_ = X.select_dtypes(include=['object', 'category']).columns.tolist()
        self.num_feats_ = X.select_dtypes(include=['number']).columns.tolist()

        # Encode categorical features
        self.encoder_ = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        X_cat = self.encoder_.fit_transform(X[self.cat_feats_])
        X_num = X[self.num_feats_].values

        # Fit one model per feature type
        self.gnb_ = GaussianNB()
        self.cnb_ = CategoricalNB(alpha=self.alpha)
        self.gnb_.fit(X_num, y, sample_weight=sample_weight)
        self.cnb_.fit(X_cat, y, sample_weight=sample_weight)

        # Store classes and shared log-prior (may diverge with non-None sample_weight)
        self.classes_ = self.gnb_.classes_
        self.log_prior_ = np.log(np.bincount(y, weights=sample_weight) 
                                  / (np.sum(sample_weight) if sample_weight is not None else len(y)))
        return self

    def _log_likelihood(
            self, 
            X: pd.DataFrame
    ) -> np.ndarray:
        # Extract log-likelihoods from each sub-model
        X_cat = self.encoder_.transform(X[self.cat_feats_])
        X_num = X[self.num_feats_].values

        # Gaussian log-likelihood: NaN -> 0 (skip missing features)
        ll_gaus_per_feat = norm.logpdf(
            x=X_num[:, np.newaxis, :],
            loc=self.gnb_.theta_, 
            scale=np.sqrt(self.gnb_.var_))  # (N, n_classes, n_num_features)
        ll_gaus = np.nan_to_num(ll_gaus_per_feat, nan=0.0).sum(axis=2)

        # Categorical log-likelihood: -1 (unseen) or NaN (missing)
        ll_cat = np.zeros((X_cat.shape[0], len(self.classes_)))
        for k, log_prob_k in enumerate(self.cnb_.feature_log_prob_):
            idx = X_cat[:, k]
            mask = np.isnan(idx) | (idx == -1)           # unknown or missing
            idx_safe = np.where(mask, 0, idx).astype(int) # safe index for lookup
            contrib = log_prob_k[:, idx_safe].T           # (N, n_classes)
            contrib[mask] = 0.0                           # zero out skipped features
            ll_cat += contrib

        return ll_gaus + ll_cat  # shape is (n_samples, n_classes)

    def predict_log_proba(
            self,
            X: pd.DataFrame,
    ) -> np.ndarray:
        return self._log_likelihood(X) + self.log_prior_ # (n_samples, n_classes)

    def predict_proba(
            self, 
            X: pd.DataFrame,
    ) -> np.ndarray:
        """
        Returns normalised posterior probabilities for each 
        instance-class pair.
        """
        # Combine likelihoods with a single shared prior
        log_prob = self.predict_log_proba(X)

        # Apply log-sum-exp trick for numerical stability
        # Shift each row by its maximum so the largest value becomes 0, avoiding underflow
        log_prob -= log_prob.max(axis=1, keepdims=True)
        prob = np.exp(log_prob)
        return prob / prob.sum(axis=1, keepdims=True)

    def predict(self, 
                X: pd.DataFrame
    ) -> np.ndarray:
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

In [11]:
# Instantiate and fit MixedNB on clean training data
model = MixedNB(alpha=1.0)
model.fit(X_train, y_train)

print("Classes:", model.classes_)
print("Log priors:", model.log_prior_)
print("Priors:", np.exp(model.log_prior_))

Classes: [0 1]
Log priors: [-0.28221372 -1.40288115]
Priors: [0.7541125 0.2458875]


In [ ]:
# Q1.1: Prior probabilities P(c)
priors = np.exp(model.log_prior_)
print(f"P(<=50K) = {priors[0]:.4f}")
print(f"P(>50K)  = {priors[1]:.4f}")

P(<=50K) = 0.7541
P(>50K)  = 0.2459


In [20]:
# Q1.2: Per-class means and standard deviations for continuous features
means = pd.DataFrame(model.gnb_.theta_, 
                     columns=model.num_feats_, 
                     index=['<=50K', '>50K'])

stds = pd.DataFrame(np.sqrt(model.gnb_.var_), 
                    columns=model.num_feats_, 
                    index=['<=50K', '>50K'])

print(f"Means:\n {means.T.round(2)}")
print(f"\nStd devs:\n {stds.T.round(2)}")


Means:
                  <=50K     >50K
age              37.05    43.94
education-num     9.63    11.59
capital-gain    157.67  3607.15
capital-loss     55.97   202.36
hours-per-week   39.43    45.64

Std devs:
                   <=50K      >50K
age               13.71     10.30
education-num      2.45      2.36
capital-gain    1017.86  13616.62
capital-loss     316.02    603.99
hours-per-week    11.91     10.40


In [ ]:
# Q1.3: Probability ratios
# feature_log_prob_[k] has shape (n_classes, n_categories_k)
# row 0 = <=50K, row 1 = >50K

records = []
for k, feat in enumerate(model.cat_feats_):
    categories = model.encoder_.categories_[k]
    log_probs = model.cnb_.feature_log_prob_[k]  # (2, n_categories)
    log_R = log_probs[1] - log_probs[0]           # log R per category value
    for v, cat in enumerate(categories):
        records.append({
            'feature': feat,
            'value': cat,
            'R': np.exp(log_R[v])
        })

ratios_df = pd.DataFrame(records)

top_per_feature = (ratios_df.loc[ratios_df.groupby('feature')['R'].idxmax()]
                   [['feature', 'value', 'R']].rename(columns={'value': '>50K value', 'R': 'R (>50K/<=50K)'}))
bot_per_feature = (ratios_df.loc[ratios_df.groupby('feature')['R'].idxmin()]
                   [['feature', 'value', 'R']].rename(columns={'value': '<=50K value', 'R': 'R (>50K/<=50K)'}))

In [23]:
print("Most predictive value per feature for >50K:")
display(top_per_feature.reset_index(drop=True).round(3))

print("\nMost predictive value per feature for <=50K:")
display(bot_per_feature.reset_index(drop=True).round(3))

Most predictive value per feature for >50K:


,feature,>50K value,R (>50K/<=50K)
0,education,Prof-school,7.959
1,marital-status,Married-AF-spouse,4.288
2,native-country,Taiwan,3.383
3,occupation,Exec-managerial,2.768
4,race,Asian-Pac-Islander,1.132
5,relationship,Wife,2.745
6,sex,Male,1.378
7,workclass,Self-emp-inc,3.251



Most predictive value per feature for <=50K:


,feature,<=50K value,R (>50K/<=50K)
0,education,9th,0.151
1,marital-status,Never-married,0.164
2,native-country,Mexico,0.183
3,occupation,Priv-house-serv,0.084
4,race,Amer-Indian-Eskimo,0.441
5,relationship,Own-child,0.052
6,sex,Female,0.397
7,workclass,Private,0.857


In [24]:
print("Top 5 most predictive of >50K (highest R):")
display(ratios_df.nlargest(5, 'R')[['feature', 'value', 'R']].round(3).reset_index(drop=True))

print("\nTop 5 most predictive of <=50K (lowest R, i.e. strongly predicts low income):")
display(ratios_df.nsmallest(5, 'R')[['feature', 'value', 'R']].round(3).reset_index(drop=True))

Top 5 most predictive of >50K (highest R):


,feature,value,R
0,education,Prof-school,7.959
1,education,Doctorate,7.135
2,marital-status,Married-AF-spouse,4.288
3,education,Masters,3.555
4,native-country,Taiwan,3.383



Top 5 most predictive of <=50K (lowest R, i.e. strongly predicts low income):


,feature,value,R
0,relationship,Own-child,0.052
1,occupation,Priv-house-serv,0.084
2,occupation,Other-service,0.135
3,relationship,Other-relative,0.145
4,education,9th,0.151


## 2. Supervised model evaluation

## 3. Extending the model with semi-supervised training

## 4. Supervised model evaluation